# TerraTree — Step 3: Model Training & Evaluation

This notebook:
1. Loads `sundarbans_training_table_v4.csv` from notebook 02
2. Cleans it up (drops non-feature columns, handles any missing values)
3. Checks feature correlation and drops redundant features (following the
   reference paper's |r| > 0.80 rule)
4. Trains a Random Forest classifier
5. Evaluates with Overall Accuracy, Kappa, confusion matrix, per-class
   precision/recall (mirrors PA/UA from the reference paper)
6. Adds SHAP explainability — which features actually drive each prediction
7. Saves the trained model for notebook 04 (applications) and notebook 05 (dashboard)

Run in Colab. This notebook does NOT need Earth Engine — everything here
runs on the CSV you already exported, using plain Python/scikit-learn.

In [ ]:
!pip install shap -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV_PATH = '/content/drive/MyDrive/terratree/sundarbans_training_table_v4.csv'
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

## Clean up

Drop Earth Engine's bookkeeping columns (`.geo`, `system:index`) — these
aren't features, just metadata from the export. Map the numeric label
codes back to readable class names for clearer plots/reports later.

In [ ]:
drop_cols = [c for c in ['.geo', 'system:index'] if c in df.columns]
df = df.drop(columns=drop_cols)

CLASS_NAMES = {0: 'Water', 1: 'Mangrove', 2: 'Other vegetation', 3: 'Bare/built'}
df['label_name'] = df['label'].map(CLASS_NAMES)

print("Class distribution:")
print(df['label_name'].value_counts())

# Drop any rows with missing feature values (can happen at cloud-masked edges)
before = len(df)
df = df.dropna()
print(f"\nDropped {before - len(df)} rows with missing values, {len(df)} remain.")

## Feature correlation check

Following the reference paper's approach: if two features are highly
correlated (|r| > 0.80), keep the one that's more informative rather
than feeding the model redundant signal.

In [ ]:
feature_cols = [c for c in df.columns if c not in ['label', 'label_name']]

corr = df[feature_cols].corr()

plt.figure(figsize=(8, 6))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.xticks(range(len(feature_cols)), feature_cols, rotation=90)
plt.yticks(range(len(feature_cols)), feature_cols)
plt.colorbar(label='Pearson r')
plt.title('Feature correlation matrix')
plt.tight_layout()
plt.show()

# Flag pairs with |r| > 0.80
high_corr_pairs = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.80:
            high_corr_pairs.append((feature_cols[i], feature_cols[j], round(r, 3)))

print("Highly correlated pairs (|r| > 0.80):")
for pair in high_corr_pairs:
    print(" ", pair)

In [ ]:
# Review the printed pairs above, then list which columns to drop.
# NDVI/NDVIre/NDre1 are the most likely candidates to overlap heavily —
# adjust this list based on what actually printed for your data.
FEATURES_TO_DROP = []  # e.g. ['NDre1'] if it's redundant with NDVIre — fill in after reviewing above

selected_features = [c for c in feature_cols if c not in FEATURES_TO_DROP]
print("Features going into the model:", selected_features)

## Train/test split and Random Forest training

70/30 split, stratified by class so the test set has proportional
representation of each class — same convention as the reference paper.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df[selected_features]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=300,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print("Model trained.")

## Evaluation: OA, Kappa, confusion matrix, per-class metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, confusion_matrix, classification_report
)

y_pred = rf.predict(X_test)

oa = accuracy_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

print(f"Overall Accuracy: {oa:.4f}")
print(f"Kappa coefficient: {kappa:.4f}\n")

print(classification_report(y_test, y_pred, target_names=[CLASS_NAMES[c] for c in sorted(y.unique())]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
labels = [CLASS_NAMES[c] for c in sorted(y.unique())]

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(len(labels)), labels, rotation=45)
plt.yticks(range(len(labels)), labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                  color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar()
plt.tight_layout()
plt.show()

## Feature importance + SHAP explainability

Two views: Random Forest's built-in importance (fast, standard), and
SHAP (slower, but shows direction and magnitude of each feature's effect
per prediction — this is what gives you a real answer if a panelist asks
"why did the model classify this pixel as mangrove?").

In [ ]:
importances = pd.Series(rf.feature_importances_, index=selected_features).sort_values(ascending=False)

plt.figure(figsize=(7, 4))
importances.plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Random Forest feature importance')
plt.tight_layout()
plt.show()

print(importances)

In [ ]:
import shap

# SHAP on a sample of the test set (full set can be slow for large data)
sample_size = min(300, len(X_test))
X_sample = X_test.sample(sample_size, random_state=42)

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_sample)

# Summary plot — one panel per class, showing which features push toward it
shap.summary_plot(shap_values, X_sample, class_names=[CLASS_NAMES[c] for c in sorted(y.unique())])

## Save the trained model

This is what notebook 04 (applications: change detection, carbon
estimate) and notebook 05 (dashboard) will load and reuse — no need to
retrain from scratch every time.

In [ ]:
import joblib
import os

os.makedirs('/content/drive/MyDrive/terratree/models', exist_ok=True)
model_path = '/content/drive/MyDrive/terratree/models/random_forest_v1.joblib'
joblib.dump({'model': rf, 'features': selected_features, 'class_names': CLASS_NAMES}, model_path)
print(f"Model saved to {model_path}")

## Next steps
- [ ] Review the correlation matrix — fill in `FEATURES_TO_DROP` above if any pairs exceed |r| > 0.80, then re-run training
- [ ] Check OA/Kappa — if accuracy looks weak (<80% OA), the most likely culprits are class imbalance or overly similar spectral signatures between 'Other vegetation' and 'Mangrove'; worth inspecting the confusion matrix for which pair is most confused
- [ ] Note in your report: labels came from GMW + ESA WorldCover, not field survey — this is a standard, legitimate substitute, but should be stated explicitly
- [ ] Move to `04_applications.ipynb` — change detection, blue-carbon estimate, biodiversity index, using this saved model